# HW#2 — การวิเคราะห์ชุดข้อมูล Titanic (Data Quality Assessment & EDA)

**รหัสนักศึกษา:** 6710503739  

เตรียมไลบรารีและโหลดข้อมูล

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy.stats import chi2_contingency, pearsonr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

df = pd.read_csv("titanic_data.csv")
df.columns = df.columns.str.strip()
print("ขนาดข้อมูล (แถว, คอลัมน์):", df.shape)
df.head()

: 

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

---
## Question 2: Data Quality Assessment
### Q2.1 การวิเคราะห์ข้อมูลสูญหาย (Missing Data Analysis)

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
})

missing

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.show()

In [ ]:
msno.heatmap(df)
plt.show()

#### สรุปผล Q2.1
คอลัมน์ที่มีค่าสูญหาย และเปอร์เซ็นต์:
- Cabin - สูญหาย 77.10% มากที่สุด
- Age - สูญหาย 19.87%
- Embarked - สูญหาย 0.22% น้อยที่สุด

เหมาะสมหรือไม่ที่จะลบคอลัมน์ Cabin ทิ้งทั้งหมด?
ตัวอักษรตัวแรกของ Cabin (C, E, B) บอก deck ซึ่งสัมพันธ์กับตำแหน่งบนเรือและโอกาสรอดชีวิต ควรสร้างข้อมูลใหม่ เช่น Deck  แล้วค่อยลบคอลัมน์ดิบทิ้งเก็บข้อมูลที่มีประโยชน์ไว้

รูปแบบที่สังเกตได้จาก heatmap:
- จาก nullity heatmap ค่าสหสัมพันธ์การหายระหว่างคอลัมน์อยู่ในระดับต่ำ นั่นคือ Age หาย ไม่ได้แปลว่า Cabin จะหายด้วย การหายของแต่ละคอลัมน์ค่อนข้างเป็นอิสระต่อกัน

### Q2.2 การตรวจจับค่าผิดปกติ (Outlier Detection)

In [ ]:
def iqr_outliers(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    return {
        'Q1': round(q1, 2), 'Q3': round(q3, 2), 'IQR': round(iqr, 2),
        'lower_bound': round(low, 2), 'upper_bound': round(high, 2),
        'n_outliers': int(mask.sum()),
        'pct_outliers': round(mask.mean() * 100, 2),
    }

summary = pd.DataFrame({col: iqr_outliers(df[col]) for col in ['Age', 'Fare', 'SibSp']}).T
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['Age', 'Fare', 'SibSp']):
    sns.boxplot(y=df[col], ax=ax, color='skyblue')
    ax.set_title(f'Box plot: {col}')
plt.tight_layout()
plt.show()

#### สรุปผล Q2.2
outlier ที่ตรวจพบด้วย IQR: Age คือสูงเกิน 64.8 ปี, Fare สูงเกิน 65.6 สูงสุด 512.3, SibSp มากกว่า 2.5 คน

2) ทำไมเลือก IQR แทน Z-score?
เพราะ Fare และ SibSp เบ้ขวาไม่ได้กระจายแบบปกติ Z-score ตั้งอยู่บนสมมติฐานการแจกแจงปกติและใช้ค่าเฉลี่ย/ส่วนเบี่ยงเบนมาตรฐาน

3) outlier ทั้งหมดเป็น "ข้อมูลเสีย" หรือไม่?
ไม่ ส่วนใหญ่เป็นค่าจริงที่ถูกต้องเช่น ผู้โดยสารอายุ 80 ปี, ค่าโดยสาร 512.33 ปอนด์ของห้องสวีทชั้นหนึ่ง ล้วนเป็นค่าที่เกิดขึ้นได้จริง ไม่ใช่ความผิดพลาดในการบันทึก

4) ควรลบ outlier ของ Fare ก่อนสร้างโมเดลทำนายหรือไม่?
ไม่ควรลบทิ้ง เพราะค่าโดยสารสูงสัมพันธ์โดยตรงกับชั้นโดยสารและฐานะ ซึ่งเป็น ตัวทำนายการรอดชีวิตได้ ทางเลือกที่ดีกว่าคือ แปลงค่า เพื่อลดอิทธิพลของค่าสุดโต่งโดยไม่ทิ้งข้อมูล

5) เทคนิคตรวจจับ outlier เพิ่มเติม (นอกจาก IQR):  
Isolation Forest - ข้อดีคือทำงานกับหลายตัวแปรพร้อมกัน จับ outlier ที่เห็นได้เฉพาะ เมื่อพิจารณาหลายมิติ ไม่ต้องสมมติการแจกแจง และปรับขนาดกับข้อมูลใหญ่ได้ดี

6) ผลกระทบของ outlier ต่อการวิเคราะห์:  
ดึงค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐาน mean ของ Fare ≈ 32 แต่ median เพียง 14.5 ทำให้ค่าสหสัมพันธ์และโมเดลเชิงเส้นไวต่อค่าเหล่านี้ จึงควรใช้ median, IQR และพิจารณาการแปลงค่าเสมอ

# Question 3: Univariate Analysis
### Q3.1 ตัวแปรเชิงตัวเลข (Numerical Variables)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f"Age (skewness = {df['Age'].skew():.2f})")
sns.histplot(df['Fare'].dropna(), bins=40, kde=True, ax=axes[1], color='indianred')
axes[1].set_title(f"Fare (skewness = {df['Fare'].skew():.2f})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['Age', 'Fare', 'SibSp']):
    sns.boxplot(y=df[col], ax=ax, color='lightgreen')
    ax.set_title(f'{col}')
plt.tight_layout()
plt.show()

#### สรุปผล Q3.1
Histogram:
- Age มีค่า skewness 0.39 - เบ้ขวาเล็กน้อย รูปทรงใกล้ระฆังคว่ำ กระจุกช่วง 20–40 ปี (mean 29.7 / median 28.0 ใกล้กัน) เกือบสมมาตร
- Fare มีค่า skewness 4.79 — เบ้ขวาอย่างมาก ผู้โดยสารส่วนใหญ่จ่ายค่าโดยสารต่ำ (mean 32.2 / median 14.5)

Box plot:
- Age การกระจายสมมาตรที่สุด มี outlier ด้านสูงเพียงไม่กี่จุด
- Fare กล่องถูกอัดที่ค่าต่ำ มี outlier ด้านสูงจำนวนมาก
- SibSp mode = 0 ผู้โดยสารส่วนใหญ่เดินทางคนเดียว ค่าที่มากกว่า 3 กลายเป็น outlier
- แนวโน้มสู่ศูนย์กลาง: median ของ Age สูงสุด (28), Fare (14.5), SibSp (0) ตามลำดับ

### Q3.2 ตัวแปรเชิงหมวดหมู่ (Categorical Variables)

In [ ]:
cat_cols = ['Survived', 'Sex', 'Pclass', 'Embarked']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(x=col, data=df, ax=ax, palette='Set2', hue=col, legend=False)
    ax.set_title(f'Distribution of {col}')
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Survived'].value_counts().plot.pie(
    autopct='%1.1f%%', labels=['Died (0)', 'Survived (1)'],
    colors=['#d95f5f', '#5f9ed9'], ax=axes[0])
axes[0].set_title('Survived'); axes[0].set_ylabel('')
df['Sex'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'], ax=axes[1])
axes[1].set_title('Sex'); axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

#### สรุปผล Q3.2
- Survived: ผู้โดยสารรอดชีวิตเพียง 38.4% คนส่วนใหญ่เสียชีวิต - ข้อมูลไม่สมดุล (imbalanced)
- Sex: เพศชายมากกว่าเพศหญิงชัดเจน
- Pclass: ผู้โดยสารชั้น 3 มากที่สุด
- Embarked: ท่าเรือ S เป็นจุดขึ้นเรือหลัก รองลงมาคือ C และ Q
- ข้อสังเกต: ความไม่สมดุลของ Survived สำคัญต่อการเลือกเมตริกประเมินโมเดลภายหลัง 

## Question 4: Bivariate Analysis – Numerical Variables
### Q4.1 การวิเคราะห์สหสัมพันธ์ (Correlation Analysis)

In [ ]:
sub = df[['Age', 'Fare']].dropna()
r, p_val = pearsonr(sub['Age'], sub['Fare'])
print(f'Pearson r = {r:.4f}')
print(f'p-value  = {p_val:.4g}')

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df, alpha=0.6, palette='coolwarm')
plt.title('Age vs Fare')
plt.show()

#### สรุปผล Q4.1
- ค่า Pearson r 0.096 - เป็นความสัมพันธ์เชิงบวกที่อ่อนมาก แปลว่าอายุกับค่าโดยสารแทบไม่มีความสัมพันธ์เชิงเส้นต่อกัน
- Scatter plot: จุดกระจายกระจุกที่ค่าโดยสารต่ำในทุกช่วงอายุ ไม่เห็นแนวโน้มเชิงเส้นชัดเจน
- สังเกตกลุ่มค่าโดยสารสูงผิดปกติซึ่งส่วนใหญ่เป็นผู้รอดชีวิต - แปลว่าค่าโดยสาร สัมพันธ์กับการรอดชีวิตมากกว่าที่อายุสัมพันธ์กับค่าโดยสาร

## Question 5: Bivariate Analysis – Categorical Variables
### Q5.1 ตารางความสัมพันธ์ (Contingency Table): Survived × Pclass

In [ ]:
ct = pd.crosstab(df['Pclass'], df['Survived'], margins=True)
ct.columns = ['Died(0)', 'Survived(1)', 'All']
print(ct)

surv_pct = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)
print('\nSurvival rate by Pclass (%):')
print(surv_pct)

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=surv_pct.index, y=surv_pct.values, palette='viridis', hue=surv_pct.index, legend=False)
for i, v in enumerate(surv_pct.values):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate (%)'); plt.xlabel('Pclass')
plt.ylim(0, 75)
plt.show()

#### สรุปผล Q5.1

อัตราการรอดชีวิตลดลงชัดเจนตามชั้นโดยสารที่ต่ำลง - ผู้โดยสารชั้น 1 มีโอกาสรอดสูงกว่าชั้น 3 ชี้ว่าชั้นโดยสารสัมพันธ์อย่างมากกับการรอดชีวิต

### Q5.2 การทดสอบทางสถิติ (Chi-Square Test of Independence)

In [ ]:
observed = pd.crosstab(df['Survived'], df['Pclass'])
chi2, p, dof, expected = chi2_contingency(observed)
print(f'Chi-square statistic = {chi2:.3f}')
print(f'p-value = {p:.3e}')
print(f'degrees of freedom = {dof}')
print('\nExpected frequencies:')
print(pd.DataFrame(expected, index=['Died', 'Survived'], columns=[1, 2, 3]).round(1))

#### สรุปผล Q5.2
สมมติฐาน:
- H0 : การรอดชีวิต และชั้นโดยสารเป็นอิสระต่อกัน
- H1 : การรอดชีวิตและชั้นโดยสารมีความสัมพันธ์กัน

ผลการทดสอบ: power(x,2) = 102.89, p-value = 4.55 × power(10,-23), degrees of freedom = 2

ข้อสรุป: เนื่องจาก p-value น้อยกว่า 0.05 อย่างมาก จึงปฏิเสธ H0 สรุปได้ว่าการรอดชีวิตกับชั้นโดยสาร มีความสัมพันธ์กันอย่างมีนัยสำคัญทางสถิติยืนยันข้อสังเกตจาก Q5.1 ว่าชั้นโดยสารเป็นปัจจัยสำคัญต่อการรอดชีวิต

---
## Question 6: Multi-variate Analysis
### Q6.1 Pairplot: Age, Fare, SibSp

In [ ]:
pair_df = df[['Age', 'Fare', 'SibSp', 'Survived']].dropna()
sns.pairplot(pair_df, vars=['Age', 'Fare', 'SibSp'], hue='Survived',
             palette='coolwarm', diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot: Age, Fare, SibSp (by Survived)', y=1.02)
plt.show()

#### สรุปผล Q6.1
- ความสัมพันธ์ระหว่างตัวแปร: ทุกคู่ (Age-Fare, Age-SibSp, Fare-SibSp) แสดงความสัมพันธ์เชิงเส้นที่อ่อนมาก ไม่มีคู่ใดสัมพันธ์กันชัดเจน 
- สัญญาณการรอดชีวิต: ในมิติ Fare ผู้ที่จ่ายค่าโดยสารสูงมีสัดส่วนรอดชีวิตมากกว่า - Fare ให้แยกแยะการรอดชีวิตได้ดีกว่า Age หรือ SibSp
- outlier ที่เห็นจาก pairplot: จุด Fare > 500, Age = 80 ปี, และ SibSp = 8 โดดออกจากกลุ่มหลักในทุกแกนที่เกี่ยวข้อง

### สรุปภาพรวม
1. คุณภาพข้อมูล: ต้องจัดการค่าสูญหายใน Cabin (77%), Age (20%), Embarked (2 แถว) และมี extreme values ใน Fare,SibSp ที่เป็นค่าจริงควรแปลงค่ามากกว่าลบทิ้ง
2. ปัจจัยที่สัมพันธ์กับการรอดชีวิต: ชั้นโดยสาร และค่าโดยสารซึ่งสะท้อนฐานะ มีความสัมพันธ์กับการรอดชีวิตอย่างมีนัยสำคัญขณะที่อายุมีบทบาทน้อยกว่า